In [0]:
%pip install pymongo pandas

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymongo]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [ ]:
import os
import sys
import urllib.parse
from datetime import datetime, timezone

# Add workspace search paths to locate src.common.logger
try:
    script_dir = os.path.dirname(__file__)
except NameError:
    script_dir = os.getcwd()

for p in [script_dir, os.path.abspath(os.path.join(script_dir, "../..")), os.path.abspath(os.path.join(script_dir, ".."))]:
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
from pymongo import MongoClient
from pyspark.sql.functions import current_timestamp, lit
from src.common.logger import CentralLogger


def ingest_mongodb_collection(
    collection_name: str,
    task_name: str,
    target_table: str,
    source_system_name: str
):
    logger = CentralLogger(spark)
    start_time = datetime.now(timezone.utc)
    pipeline_name = "bronze_mongodb_ingestion"

    try:
        # 1. Retrieve password securely from Secret Scope
        raw_password = dbutils.secrets.get(
            scope="maven-secrets",
            key="mongodb-password"
        )

        # 2. Construct Mongo URI
        encoded_password = urllib.parse.quote_plus(raw_password)
        cluster_host = "cluster0.sfjjdaf.mongodb.net"

        mongo_uri = (
            f"mongodb+srv://maven_admin:{encoded_password}"
            f"@{cluster_host}/?retryWrites=true&w=majority"
        )

        # 3. Connect via PyMongo
        client = MongoClient(mongo_uri)
        db = client["mavenmarket"]
        collection = db[collection_name]

        # Fetch records
        records = list(collection.find())

        if not records:
            raise ValueError(
                f"MongoDB collection 'mavenmarket.{collection_name}' is empty!"
            )

        # Drop BSON ObjectIDs
        for record in records:
            record.pop("_id", None)

        # 4. Convert Pandas DataFrame to PySpark DataFrame
        pandas_df = pd.DataFrame(records)
        df_raw = spark.createDataFrame(pandas_df)

        # 5. Append Bronze audit metadata columns
        df_bronze = (
            df_raw
            .withColumn("_ingestion_timestamp", current_timestamp())
            .withColumn("_source_system", lit(source_system_name))
        )

        records_processed = df_bronze.count()

        # 6. Overwrite Unity Catalog Delta Bronze Table
        (
            df_bronze.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_table)
        )

        # 7. Log Success
        logger.log_execution(
            pipeline_name=pipeline_name,
            task_name=task_name,
            execution_status="SUCCESS",
            start_time=start_time,
            records_processed=records_processed,
            records_failed=0
        )

        print(
            f"✅ Ingestion complete! Successfully logged and wrote "
            f"{records_processed} records to '{target_table}'."
        )

    except Exception as e:
        # Log Failure
        logger.log_execution(
            pipeline_name=pipeline_name,
            task_name=task_name,
            execution_status="FAILED",
            start_time=start_time,
            records_processed=0,
            records_failed=0,
            error=e
        )

        raise e


# ==========================================
# Execute Ingestion for Products
# ==========================================
ingest_mongodb_collection(
    collection_name="products",
    task_name="raw_products",
    target_table="dbx_maven_market.bronze.raw_products",
    source_system_name="MongoDB Atlas - Products Master"
)

# ==========================================
# Execute Ingestion for Customers
# ==========================================
ingest_mongodb_collection(
    collection_name="customers",
    task_name="raw_customers",
    target_table="dbx_maven_market.bronze.raw_customers",
    source_system_name="MongoDB Atlas - Customers Master"
)

In [0]:
%sql
SELECT 'raw_products' AS source_table, COUNT(*) AS row_count FROM dbx_maven_market.bronze.raw_products
UNION ALL
SELECT 'raw_customers' AS source_table, COUNT(*) AS row_count FROM dbx_maven_market.bronze.raw_customers;

product_id,product_name,product_sku,_ingestion_timestamp,_source_system
781,Cormorant Copper Pot Scrubber,43908330798,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
782,Cormorant Economy Toilet Brush,99983676075,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
783,Cormorant Scissors,33419412970,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
784,Cormorant Large Sponge,69949565529,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
785,Cormorant Screw Driver,86525198817,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
786,Footnote Extra Lean Hamburger,92293117642,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
787,Footnote Seasoned Hamburger,54204387068,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
788,Excel Monthly Sports Magazine,29210072159,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
789,Excel Monthly Auto Magazine,50359107851,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
790,Excel Monthly Computer Magazine,84602559208,2026-10-06T09:00:06.977435Z,MongoDB Atlas - Products Master
